In [3]:
import random
import time
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_ROOT = Path("/content/S2Looking_extracted/S2Looking")
MODEL_DIR = Path("/content/drive/MyDrive/SpaceNetra/models")

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = MODEL_DIR / "spacenetra_s2looking_unet.pth"

IMAGE_SIZE = 256
MAX_TRAIN_PAIRS = 500
MAX_VAL_PAIRS = 100

BATCH_SIZE = 4
EPOCHS = 3
LEARNING_RATE = 1e-3

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

assert DATA_ROOT.exists(), f"Dataset missing: {DATA_ROOT}"
assert device.type == "cuda", (
    "GPU is not available. Enable a GPU runtime before training."
)


# ============================================================
# 2. DATASET LOADER
# ============================================================

class S2LookingDataset(Dataset):

    def __init__(self, split, max_pairs=None, seed=42):

        self.root = DATA_ROOT / split

        image1_dir = self.root / "Image1"
        image2_dir = self.root / "Image2"
        label_dir = self.root / "label"

        if not all(
            folder.exists()
            for folder in [image1_dir, image2_dir, label_dir]
        ):
            raise FileNotFoundError(
                f"Missing Image1, Image2, or label directory "
                f"for split: {split}"
            )

        ids1 = {p.stem for p in image1_dir.glob("*.png")}
        ids2 = {p.stem for p in image2_dir.glob("*.png")}
        ids_label = {p.stem for p in label_dir.glob("*.png")}

        # Use only complete, correctly paired samples.
        self.ids = sorted(ids1 & ids2 & ids_label)

        if max_pairs is not None and len(self.ids) > max_pairs:
            rng = random.Random(seed)
            self.ids = sorted(
                rng.sample(self.ids, max_pairs)
            )

        self.image1_dir = image1_dir
        self.image2_dir = image2_dir
        self.label_dir = label_dir

        print(
            f"{split}: {len(self.ids)} matched samples"
        )

        if not self.ids:
            raise RuntimeError(
                f"No matching samples found in {self.root}"
            )

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):

        sample_id = self.ids[index]

        image1_path = self.image1_dir / f"{sample_id}.png"
        image2_path = self.image2_dir / f"{sample_id}.png"
        label_path = self.label_dir / f"{sample_id}.png"

        with Image.open(image1_path) as im:
            image1 = im.convert("RGB").resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR
            )

        with Image.open(image2_path) as im:
            image2 = im.convert("RGB").resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.BILINEAR
            )

        with Image.open(label_path) as im:
            mask = im.convert("L").resize(
                (IMAGE_SIZE, IMAGE_SIZE),
                Image.Resampling.NEAREST
            )

        image1 = np.asarray(
            image1, dtype=np.float32
        ) / 255.0

        image2 = np.asarray(
            image2, dtype=np.float32
        ) / 255.0

        # Binary mask: change = 1, background = 0.
        mask = (
            np.asarray(mask, dtype=np.uint8) > 0
        ).astype(np.float32)

        # Concatenate both RGB images:
        # 3 channels + 3 channels = 6 channels.
        combined = np.concatenate(
            [image1, image2], axis=2
        )

        combined = torch.from_numpy(
            combined.transpose(2, 0, 1).copy()
        )

        mask = torch.from_numpy(mask.copy()).unsqueeze(0)

        return combined, mask


# ============================================================
# 3. CREATE TRAINING AND VALIDATION LOADERS
# ============================================================

train_dataset = S2LookingDataset(
    split="train",
    max_pairs=MAX_TRAIN_PAIRS,
    seed=SEED
)

val_dataset = S2LookingDataset(
    split="val",
    max_pairs=MAX_VAL_PAIRS,
    seed=SEED
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))


# ============================================================
# 4. BUILD A SMALL U-NET
# ============================================================

def conv_block(in_channels, out_channels):

    return nn.Sequential(
        nn.Conv2d(
            in_channels, out_channels,
            kernel_size=3, padding=1
        ),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),

        nn.Conv2d(
            out_channels, out_channels,
            kernel_size=3, padding=1
        ),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )


class SpaceNetraUNet(nn.Module):

    def __init__(self):

        super().__init__()

        self.pool = nn.MaxPool2d(2)

        self.enc1 = conv_block(6, 16)
        self.enc2 = conv_block(16, 32)
        self.enc3 = conv_block(32, 64)
        self.enc4 = conv_block(64, 128)

        self.middle = conv_block(128, 256)

        self.up4 = nn.ConvTranspose2d(
            256, 128, kernel_size=2, stride=2
        )
        self.dec4 = conv_block(256, 128)

        self.up3 = nn.ConvTranspose2d(
            128, 64, kernel_size=2, stride=2
        )
        self.dec3 = conv_block(128, 64)

        self.up2 = nn.ConvTranspose2d(
            64, 32, kernel_size=2, stride=2
        )
        self.dec2 = conv_block(64, 32)

        self.up1 = nn.ConvTranspose2d(
            32, 16, kernel_size=2, stride=2
        )
        self.dec1 = conv_block(32, 16)

        # One output channel: probability of change.
        self.output = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):

        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        middle = self.middle(self.pool(e4))

        d4 = self.dec4(
            torch.cat([self.up4(middle), e4], dim=1)
        )

        d3 = self.dec3(
            torch.cat([self.up3(d4), e3], dim=1)
        )

        d2 = self.dec2(
            torch.cat([self.up2(d3), e2], dim=1)
        )

        d1 = self.dec1(
            torch.cat([self.up1(d2), e1], dim=1)
        )

        return self.output(d1)


model = SpaceNetraUNet().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

bce_loss = nn.BCEWithLogitsLoss()


# ============================================================
# 5. LOSS AND VALIDATION METRICS
# ============================================================

def combined_loss(logits, targets):

    bce = bce_loss(logits, targets)

    probs = torch.sigmoid(logits)

    dims = (1, 2, 3)

    intersection = (probs * targets).sum(dim=dims)

    dice = (
        2.0 * intersection + 1.0
    ) / (
        probs.sum(dim=dims)
        + targets.sum(dim=dims)
        + 1.0
    )

    return bce + (1.0 - dice).mean()


@torch.no_grad()
def evaluate(model, loader):

    model.eval()

    tp = 0
    fp = 0
    fn = 0
    total_loss = 0.0

    for images, masks in loader:

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        logits = model(images)

        total_loss += combined_loss(
            logits, masks
        ).item()

        predictions = torch.sigmoid(logits) >= 0.5
        actual = masks >= 0.5

        tp += (predictions & actual).sum().item()
        fp += (predictions & ~actual).sum().item()
        fn += (~predictions & actual).sum().item()

    eps = 1e-8

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)

    f1 = (
        2 * precision * recall
        / (precision + recall + eps)
    )

    iou = tp / (tp + fp + fn + eps)

    return {
        "loss": total_loss / max(len(loader), 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou
    }


# ============================================================
# 6. TRAIN AND SAVE THE CHECKPOINT
# ============================================================

best_f1 = -1.0

print("\nStarting training...\n")

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0
    start_time = time.time()

    for step, (images, masks) in enumerate(train_loader, start=1):

        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(images)
        loss = combined_loss(logits, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if step % 20 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {step}/{len(train_loader)} | "
                f"Loss {loss.item():.4f}"
            )

    train_loss = running_loss / max(len(train_loader), 1)

    metrics = evaluate(model, val_loader)

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch + 1} complete")
    print(f"Training loss: {train_loss:.4f}")
    print(f"Validation loss: {metrics['loss']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    print(f"F1: {metrics['f1']:.4f}")
    print(f"IoU: {metrics['iou']:.4f}")
    print(f"Time: {elapsed:.1f} seconds\n")

    # Keep the best validation-F1 checkpoint.
    if metrics["f1"] > best_f1:

        best_f1 = metrics["f1"]

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "image_size": IMAGE_SIZE,
                "input_channels": 6,
                "output_channels": 1,
                "epoch": epoch + 1,
                "validation_metrics": metrics,
                "training_pairs": len(train_dataset),
                "validation_pairs": len(val_dataset),
                "seed": SEED
            },
            CHECKPOINT_PATH
        )

        print("Saved improved checkpoint:", CHECKPOINT_PATH)

print("\nTraining finished.")
print("Model checkpoint:", CHECKPOINT_PATH)
print("Best validation F1:", best_f1)

assert CHECKPOINT_PATH.exists(), "Checkpoint was not saved."

Device: cpu


AssertionError: Dataset missing: /content/S2Looking_extracted/S2Looking